# Health Insurance Cost Prediction

**Course:** Predictive Analytics (CSE 3141) — Manipal University Jaipur
**Author:** Bhoomi Tiwari

A multiple linear regression that predicts annual medical costs billed to health
insurance, from six policyholder attributes.

The output is a continuous amount of money, so this is a **regression** problem —
not classification. There is no probability and no decision threshold.

Run the cells top to bottom. The dataset is included at `../data/insurance.csv`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

df = pd.read_csv('../data/insurance.csv')
df.head()

In [ ]:
print('shape      :', df.shape)
print('missing    :', int(df.isna().sum().sum()))
print(f'avg charge : ${df.charges.mean():,.0f}')
print()
print(df.describe().round(2))

## What to notice before modelling

Two things in this data shape the model:

1. **The target is heavily skewed.** Charges run from about $1,122 to $63,770 — a
   57-fold spread. A small number of very expensive people pull the mean upward.
2. **Smoking is not a flat surcharge.** A smoker at BMI 25 and a smoker at BMI 40
   are not the same risk. This is what forces an interaction term later.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

ax[0].hist(df.charges, bins=40, color='#4C4941', edgecolor='white')
ax[0].set_title('Distribution of charges')
ax[0].set_xlabel('Annual charge (USD)')
ax[0].set_ylabel('Count')

for label, colour in [('no', '#4C4941'), ('yes', '#B3261E')]:
    sub = df[df.smoker == label]
    ax[1].scatter(sub.bmi, sub.charges, s=9, alpha=.45, color=colour, label=f'smoker={label}')
ax[1].set_title('Charges against BMI, split by smoking')
ax[1].set_xlabel('BMI')
ax[1].set_ylabel('Annual charge (USD)')
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()

print(f'mean charge, non-smoker : ${df[df.smoker=="no"].charges.mean():,.0f}')
print(f'mean charge, smoker     : ${df[df.smoker=="yes"].charges.mean():,.0f}')

## Feature engineering

Categoricals become 0/1 dummies. Region has four categories, so it needs three
dummies — `northeast` is the baseline and is represented by all three being zero.

The interaction term `smoker × bmi` is added by hand.


In [ ]:
FEATURES = ['age','bmi','children','male','smoker','nw','se','sw','smoker_x_bmi']

def build_features(d):
    X = pd.DataFrame({
        'age':      d['age'].astype(float),
        'bmi':      d['bmi'].astype(float),
        'children': d['children'].astype(float),
        'male':     (d['sex'] == 'male').astype(float),
        'smoker':   (d['smoker'] == 'yes').astype(float),
        'nw':       (d['region'] == 'northwest').astype(float),
        'se':       (d['region'] == 'southeast').astype(float),
        'sw':       (d['region'] == 'southwest').astype(float),
    })
    X['smoker_x_bmi'] = X['smoker'] * X['bmi']
    return X[FEATURES]

X, y = build_features(df), df['charges']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)
print('train', len(X_train), '/ test', len(X_test))
X.head()

## Baseline: linear regression without the interaction

This establishes what the interaction term is actually buying us.


In [ ]:
def score(y_true, y_pred, label):
    print(f'{label:<28} R2={r2_score(y_true, y_pred):.3f}  '
          f'MAE=${mean_absolute_error(y_true, y_pred):,.0f}  '
          f'RMSE=${np.sqrt(mean_squared_error(y_true, y_pred)):,.0f}')

plain = LinearRegression().fit(X_train.drop(columns=['smoker_x_bmi']), y_train)
score(y_test, plain.predict(X_test.drop(columns=['smoker_x_bmi'])), 'linear (no interaction)')

## The model: linear regression + smoker × BMI

And a gradient boosting model as an accuracy ceiling to check whether the simple
model is leaving much on the table.


In [ ]:
model = LinearRegression().fit(X_train, y_train)
score(y_test, model.predict(X_test), 'linear + smoker x BMI')

gbm = GradientBoostingRegressor(random_state=42).fit(X_train, y_train)
score(y_test, gbm.predict(X_test), 'gradient boosting')

cv = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2').mean()
print(f'\n5-fold CV R2 (linear)        {cv:.3f}')

Gradient boosting is only marginally better (0.879 against 0.865) and is not
interpretable. The linear model was kept deliberately: every coefficient can be
read, defended and explained to a non-technical audience.


In [ ]:
final = LinearRegression().fit(X, y)
print(f'intercept: ${final.intercept_:,.2f}\n')
for name, coef in zip(FEATURES, final.coef_):
    print(f'  {name:<14} {coef:+12,.2f}')

print('\nThe smoking effect must be read as one term:')
print('  cost of smoking = %.2f + %.2f x bmi' % (final.coef_[4], final.coef_[8]))
for b in (25, 30, 32, 40):
    print(f'    at BMI {b} -> ${final.coef_[4] + final.coef_[8]*b:,.0f}')

## Conclusion

A multiple linear regression with one interaction term explains **86.5%** of the
variation in medical charges, with a mean absolute error of about **$2,757** on
held-out data.

The single most important finding is that smoking cannot be modelled as a flat
surcharge. Its cost compounds with body mass — the interaction term lifts R² from
0.784 to 0.865, and that one change is most of the model's value.

### Limitations

- **Wrong country.** These are US charges in dollars; they say nothing about Indian premiums.
- **No medical history.** The six inputs exclude diagnoses, prescriptions and claims — the things real underwriting uses.
- **Assumes a straight line.** Outside the training range (BMI 16–53) the model is extrapolating. It implies smoking becomes *cheaper* below BMI 14.15, which is nonsense.
- **Small and observational.** 1,338 rows demonstrates a method; it does not establish causation.

This is a teaching model, not a pricing tool.
